In [1]:
import numpy as np
import os
import itertools
import logging
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
import torch.nn.functional as F
from argparse import ArgumentParser
from torch.distributions import MultivariateNormal
import sys
sys.path.append("../")
import pymbar
from config import get_cfg_defaults
from nf.flows import *
from nf.models import NormalizingFlowModel
from nf.base import EinsteinCrystal
import nf.utils as util
import random
%load_ext autoreload
%autoreload 2

/Users/sherryli/opt/miniconda3/envs/sherry/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def setup_model(cfg):
    if cfg.dataset.rho is not None:
        B=(cfg.dataset.nparticles/(8*cfg.dataset.rho))**(1/3)
    else:
        B=cfg.dataset.ncellx*cfg.dataset.cell_len/2
    boxlength=2*B
    N=cfg.dataset.nparticles*3
    logging.basicConfig(level=logging.DEBUG)
    logger = logging.getLogger(__name__)  
    if cfg.prior.type=="lattice":
        prior = EinsteinCrystal(cfg.prior.lattice_dir, alpha=cfg.prior.alpha,device=cfg.device)
    elif cfg.prior.type=="normal":
        prior = MultivariateNormal(torch.zeros(N).to(cfg.device), 0.5*torch.eye(N).to(cfg.device))
    if cfg.flow.type=="RealNVP":
        flows = [eval(cfg.flow.type)(dim=N,hidden_dim=cfg.flow.hidden_dim) for _ in range(cfg.flow.nlayers)]
    elif cfg.flow.type=="NSF_AR":
        flows = [eval(cfg.flow.type)(dim=N, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim) for _ in range(cfg.flow.nlayers)]
    elif cfg.flow.type=="NSF_CL":
        x = [[0],[1],[2],[0,1],[1,2],[0,2]]
        mask= sum([x for _ in range(cfg.flow.nlayers//6+1)], [])[:cfg.flow.nlayers]
        flows = [eval(cfg.flow.type)(size=cfg.dataset.nparticles,dim=3, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim, mask=mask[i],device=cfg.device) for i in range(cfg.flow.nlayers)]
    model = NormalizingFlowModel(prior, flows,cfg.device).to(cfg.device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.train_parameters.learning_rate)
    #scheduler = torch. optim.lr_scheduler.ExponentialLR(optimizer, cfg.train_parameters.lr_scheduler_gamma)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.train_parameters.max_epochs)
    training_data = util.load_position(cfg.dataset.input_dir).to(cfg.device)
    with open(cfg.output.pos_dir, 'w'):
        pass
    if not(os.path.exists(cfg.output.model_dir)):
        os.mkdir(cfg.output.model_dir)
    return model,optimizer,scheduler,training_data,logger,boxlength

In [3]:
def generate_data(cfg,batchsize):
    nparticles=cfg.dataset.nparticles
    which_dist=torch.Tensor([random.randint(0,1) for _ in range(batchsize*nparticles)]).unsqueeze(-1)
    samples1=MultivariateNormal(torch.Tensor([0.5,0.5,0.5]), 0.1*torch.eye(3)).sample((batchsize*nparticles,))
    samples2=MultivariateNormal(torch.Tensor([-0.5,-0.5,-0.5]), 0.05*torch.eye(3)).sample((batchsize*nparticles,))
    return (samples1*which_dist+samples2*(1-which_dist)).reshape(batchsize,nparticles*3).to(cfg.device)

def evaluate_logp(cfg,x):
    x=x.reshape(-1,3)
    dist1=MultivariateNormal(torch.Tensor([0.5,0.5,0.5]), 0.1*torch.eye(3))
    dist2=MultivariateNormal(torch.Tensor([-0.5,-0.5,-0.5]), 0.05*torch.eye(3))
    logprob=torch.log(0.5*torch.exp(dist1.log_prob(x))+0.5*torch.exp(dist2.log_prob(x)))
    return torch.sum(logprob.reshape(-1,cfg.dataset.nparticles),axis=1)


In [4]:
def train(cfg,model,optimizer,scheduler,training_data,logger,boxlength):
    with open("base.xyz", 'w'):
            pass
    losses=[]
    max_logprob=140
    lamb=0.3
    for i in range(cfg.train_parameters.max_epochs):
        optimizer.zero_grad()
        x = generate_data(cfg,cfg.train_parameters.batch_size)
        z, prior_logprob, log_det = model(x)
        util.write_coord("base.xyz",z.detach(),cfg.dataset.nparticles,boxlength)
        #x1, log_det1 =model.inverse(z)
        #print("check the map is invertible", torch.linalg.norm(x1-x), torch.linalg.norm(log_det1+log_det))
        logprob = prior_logprob + log_det
        forward_loss=-torch.mean(logprob)+torch.mean(evaluate_logp(cfg,x))
        if i>1000:
            loss = forward_loss*lamb + (1-lamb)*reverseKL(cfg,model, cfg.train_parameters.batch_size)
        else:
            loss = forward_loss
        losses.append(loss.mean().data)
        loss.backward()
        optimizer.step()
        scheduler.step()
        if i % cfg.train_parameters.output_freq == 0:
            logger.info(f"Iter: {i}\t" +
                        f"Loss: {loss.mean().data:.2f}\t" +
                        f"Logprob: {logprob.mean().data:.2f}\t" +
                        f"Prior: {prior_logprob.mean().data:.2f}\t" +
                        f"LogDet: {log_det.mean().data:.2f}")
            samples,_,z = model.sample(1)
            util.write_coord(cfg.output.pos_dir,samples,cfg.dataset.nparticles,boxlength)
            if (i>1200) and (-forward_loss>max_logprob):
                max_logprob=-forward_loss
                torch.save({"model":model.state_dict(),"optim": optimizer.state_dict(),
                            "loss":losses},cfg.output.model_dir+cfg.dataset.name+'%d.pth'% (i//cfg.train_parameters.output_freq))

In [5]:
def read_input(dir):
    cfg = get_cfg_defaults()
    cfg.merge_from_file(dir)
    cfg.freeze()
    print(cfg)
    return cfg

In [6]:
def reverseKL(cfg,model,nsamples):
    z = model.prior.sample((nsamples,))
    x, log_det = model.inverse(z)
    log_prob = model.prior.log_prob(z)-log_det
    return -torch.mean(evaluate_logp(cfg,x))+torch.mean(log_prob)

In [25]:
cfg=read_input("input/mixed_gauss.yaml")
model,optimizer,scheduler,training_data,logger,boxlength = setup_model(cfg)

dataset:
  cell_len: 1
  input_dir: structures/lj.xyz
  kT: 1.0
  name: mGauss
  ncellx: 2
  ncelly: 2
  ncellz: 2
  nparticles: 10
  rho: None
device: cpu
flow:
  hidden_dim: 120
  nlayers: 4
  nsplines: 20
  type: NSF_AR
output:
  model_dir: saved_models/
  pos_dir: ./gauss_positions_during_training.xyz
prior:
  alpha: 100
  lattice_dir: structures/ref.xyz
  type: normal
train_parameters:
  batch_size: 10
  learning_rate: 0.0005
  lr_scheduler_gamma: 0.999
  max_epochs: 8000
  output_freq: 100


In [8]:
x=generate_data(cfg,10)
evaluate_logp(cfg,x)

tensor([ -8.0659, -17.7097,  -7.6506, -12.2097, -17.5564,  -2.6379, -11.3147,
         -8.3795, -15.7583, -11.9626])

In [26]:
train(cfg,model,optimizer,scheduler,training_data,logger,boxlength)

INFO:__main__:Iter: 0	Loss: 17.32	Logprob: -27.44	Prior: -27.05	LogDet: -0.39
INFO:__main__:Iter: 100	Loss: 13.06	Logprob: -22.09	Prior: -25.07	LogDet: 2.98
INFO:__main__:Iter: 200	Loss: 10.32	Logprob: -19.32	Prior: -24.78	LogDet: 5.46
INFO:__main__:Iter: 300	Loss: 8.70	Logprob: -16.90	Prior: -24.44	LogDet: 7.55
INFO:__main__:Iter: 400	Loss: 6.07	Logprob: -19.77	Prior: -26.62	LogDet: 6.85
INFO:__main__:Iter: 500	Loss: 8.04	Logprob: -16.97	Prior: -25.96	LogDet: 8.99
INFO:__main__:Iter: 600	Loss: 6.72	Logprob: -16.18	Prior: -25.72	LogDet: 9.54
INFO:__main__:Iter: 700	Loss: 6.34	Logprob: -16.08	Prior: -25.52	LogDet: 9.44
INFO:__main__:Iter: 800	Loss: 6.07	Logprob: -16.21	Prior: -25.03	LogDet: 8.82
INFO:__main__:Iter: 900	Loss: 6.52	Logprob: -15.99	Prior: -25.08	LogDet: 9.09
INFO:__main__:Iter: 1000	Loss: 6.27	Logprob: -14.91	Prior: -24.91	LogDet: 10.01
INFO:__main__:Iter: 1100	Loss: 24.87	Logprob: -18.86	Prior: -26.97	LogDet: 8.11
INFO:__main__:Iter: 1200	Loss: 26.95	Logprob: -15.69	Prior

KeyboardInterrupt: 

In [ ]:
def generate_from_nf(model, prior, nsamples=50):
    #x, log_det ,z = model.sample(nsamples)
    z=prior.sample((nsamples,))
    x, log_det = model.inverse(z)
    log_px=prior.log_prob(z)-log_det
    return x.data, log_px.data

In [ ]:
traj0,q00=generate_from_nf(model,model.prior, nsamples=800)
q00=q00.cpu().numpy()
traj0=traj0.cpu().reshape(-1,cfg.dataset.nparticles,3)
q01=evaluate_logp(cfg,traj0).detach().cpu().numpy()
Q=[]
Q.append(np.transpose(np.vstack((q00,q01))))
traj1=generate_data(cfg,800)
z,_,log_det=model.forward(traj1.reshape(len(traj1),-1))
q10=model.prior.log_prob(z)-log_det
q10=q10.detach().cpu().numpy()
q11=evaluate_logp(cfg,traj1)
q11=q11.detach().cpu().numpy()
Q.append(np.transpose(np.vstack((q10,q11))))
with open("Q0.dat","w"):
    pass
with open("Q0.dat", "ab") as f:
    np.savetxt(f, Q[0])
np.savetxt("Q1.dat",Q[1])

In [ ]:
Nk=np.array([len(Q[0]),len(Q[1])])
u=np.vstack((-Q[0],-Q[1])).transpose()
print(u.shape)
mbar=pymbar.mbar.MBAR(u,Nk)
normconst=mbar.getFreeEnergyDifferences(return_dict=True)
print(normconst)

(2, 1600)
{'Delta_f': array([[ 0.        , -9.17607162],
       [ 9.17607162,  0.        ]]), 'dDelta_f': array([[ 0.        , 12.47914765],
       [12.78916033,  0.        ]])}
